In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.rebuild_comparison import build_rebuild_comparison_stage

In [ ]:
engine = get_engine_from_env()

----

In [ ]:
comparison_result = build_rebuild_comparison_stage(
    engine=engine,
    schema="capstone",
    premelt_table="synthetic_observations_premelt_stage",
    rebuilt_table="synthetic_sensor_observations_rebuilt_stage",
    target_table="synthetic_sensor_rebuild_comparison_stage",
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    n_sensors=52,
    float_tolerance=1e-9,
    observation_window_size=2500,
)

In [ ]:

comparison_result

----

In [ ]:
summary_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS comparison_row_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = TRUE) AS all_match_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = FALSE) AS mismatch_count,
        MAX(comparison_mismatch_count) AS max_mismatch_count
    FROM capsone.synthetic_sensor_rebuild_comparison_stage
    """
)


In [ ]:

display(summary_dataframe)

----

In [ ]:
mismatch_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        comparison_mismatch_count,
        comparison_notes,
        exists_in_original,
        exists_in_rebuilt,
        rebuilt__rebuild_sensor_count,
        rebuilt__rebuild_is_complete
    FROM capstone.synthetic_sensor_rebuild_comparison_stage
    WHERE comparison_all_fields_match = FALSE
    ORDER BY observation_index
    LIMIT 50
    """
)


In [ ]:


display(mismatch_dataframe)

-----

In [ ]:
observation_to_check = 1

detail_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM capstone.synthetic_sensor_rebuild_comparison_stage
    WHERE observation_index = {int(observation_to_check)}
    """
)


In [ ]:


display(detail_dataframe)